In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# =========================================================
# ATTENTION GATE
# =========================================================

class AttentionGate(nn.Module):

    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        self.Wg = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.Wx = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, g):

        g1 = self.Wg(g)
        x1 = self.Wx(x)

        psi = self.relu(g1 + x1)
        psi = self.psi(psi)

        return x * psi


# =========================================================
# ATTENTION U-NET WITH EFFICIENTNET-B0 ENCODER
# =========================================================

class UNet(nn.Module):

    def __init__(self, n_class=1, dropout_p=0.5):

        super().__init__()

        # =================================================
        # PRETRAINED ENCODER
        # =================================================

        self.encoder = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            features_only=True,
            in_chans=1,
            out_indices=(0, 1, 2, 3, 4)
        )

        # EfficientNet-B0 feature channels
        # [16, 24, 40, 112, 320]

        # =================================================
        # DROPOUT
        # =================================================

        self.drop1 = nn.Dropout2d(p=dropout_p)         # deepest block
        self.drop2 = nn.Dropout2d(p=dropout_p)
        self.drop3 = nn.Dropout2d(p=dropout_p * 0.67)  # ~0.2
        self.drop4 = nn.Dropout2d(p=dropout_p * 0.33)  # ~0.1 — lightest, near output

        # =================================================
        # DECODER
        # =================================================

        # -------------------------
        # Decoder Block 1
        # -------------------------

        self.up1 = nn.ConvTranspose2d(320, 112, kernel_size=2, stride=2)

        self.att1 = AttentionGate(F_g=112, F_l=112, F_int=56)

        self.d11 = nn.Conv2d(224, 112, kernel_size=3, padding=1)
        self.bn_d11 = nn.BatchNorm2d(112)

        self.d12 = nn.Conv2d(112, 112, kernel_size=3, padding=1)
        self.bn_d12 = nn.BatchNorm2d(112)

        # -------------------------
        # Decoder Block 2
        # -------------------------

        self.up2 = nn.ConvTranspose2d(112, 40, kernel_size=2, stride=2)

        self.att2 = AttentionGate(F_g=40, F_l=40, F_int=20)

        self.d21 = nn.Conv2d(80, 40, kernel_size=3, padding=1)
        self.bn_d21 = nn.BatchNorm2d(40)

        self.d22 = nn.Conv2d(40, 40, kernel_size=3, padding=1)
        self.bn_d22 = nn.BatchNorm2d(40)

        # -------------------------
        # Decoder Block 3
        # -------------------------

        self.up3 = nn.ConvTranspose2d(40, 24, kernel_size=2, stride=2)

        self.att3 = AttentionGate(F_g=24, F_l=24, F_int=12)

        self.d31 = nn.Conv2d(48, 24, kernel_size=3, padding=1)
        self.bn_d31 = nn.BatchNorm2d(24)

        self.d32 = nn.Conv2d(24, 24, kernel_size=3, padding=1)
        self.bn_d32 = nn.BatchNorm2d(24)

        # -------------------------
        # Decoder Block 4
        # -------------------------

        self.up4 = nn.ConvTranspose2d(24, 16, kernel_size=2, stride=2)

        self.att4 = AttentionGate(F_g=16, F_l=16, F_int=8)

        self.d41 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.bn_d41 = nn.BatchNorm2d(16)

        self.d42 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn_d42 = nn.BatchNorm2d(16)

        # =================================================
        # FINAL UPSAMPLING
        # =================================================

        self.final_up = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2)

        self.d_final1 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn_final1 = nn.BatchNorm2d(16)
        self.d_final2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn_final2 = nn.BatchNorm2d(16)

        # =================================================
        # OUTPUT
        # =================================================

        self.outconv = nn.Conv2d(16, n_class, kernel_size=1)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):

        # =================================================
        # ENCODER
        # =================================================

        features = self.encoder(x)

        x1 = features[0]   # 16 channels
        x2 = features[1]   # 24 channels
        x3 = features[2]   # 40 channels
        x4 = features[3]   # 112 channels
        x5 = features[4]   # 320 channels

        # =================================================
        # DECODER 1
        # =================================================

        d1 = self.up1(x5)
        d1 = F.interpolate(d1, size=x4.shape[-2:], mode='bilinear', align_corners=False)
        x4_att = self.att1(x4, d1)
        d1 = torch.cat([d1, x4_att], dim=1)

        d1 = self.relu(self.bn_d11(self.d11(d1)))
        d1 = self.relu(self.bn_d12(self.d12(d1)))
        d1 = self.drop1(d1)

        # =================================================
        # DECODER 2
        # =================================================

        d2 = self.up2(d1)
        d2 = F.interpolate(d2, size=x3.shape[-2:], mode='bilinear', align_corners=False)
        x3_att = self.att2(x3, d2)
        d2 = torch.cat([d2, x3_att], dim=1)

        d2 = self.relu(self.bn_d21(self.d21(d2)))
        d2 = self.relu(self.bn_d22(self.d22(d2)))
        d2 = self.drop2(d2)

        # =================================================
        # DECODER 3
        # =================================================

        d3 = self.up3(d2)
        d3 = F.interpolate(d3, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        x2_att = self.att3(x2, d3)
        d3 = torch.cat([d3, x2_att], dim=1)

        d3 = self.relu(self.bn_d31(self.d31(d3)))
        d3 = self.relu(self.bn_d32(self.d32(d3)))
        d3 = self.drop3(d3)

        # =================================================
        # DECODER 4
        # =================================================

        d4 = self.up4(d3)
        d4 = F.interpolate(d4, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        x1_att = self.att4(x1, d4)
        d4 = torch.cat([d4, x1_att], dim=1)

        d4 = self.relu(self.bn_d41(self.d41(d4)))
        d4 = self.relu(self.bn_d42(self.d42(d4)))
        d4 = self.drop4(d4)

        # =================================================
        # FINAL UPSAMPLING
        # =================================================

        d4 = self.final_up(d4)
        d4 = self.relu(self.bn_final1(self.d_final1(d4)))
        d4 = self.relu(self.bn_final2(self.d_final2(d4)))

        # =================================================
        # OUTPUT
        # =================================================

        out = self.outconv(d4)

        return out

In [2]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import kagglehub
import random


import random
import cv2
import numpy as np

def augment(image, mask):

    # Horizontal flip
    if random.random() > 0.3:
        image = cv2.flip(image, 1)
        mask  = cv2.flip(mask, 1)

    # Vertical flip
    if random.random() > 0.3:
        image = cv2.flip(image, 0)
        mask  = cv2.flip(mask, 0)

    # Rotation
    if random.random() > 0.3:
        angle = random.uniform(-30, 30)
        h, w  = image.shape[:2]
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR)
        mask  = cv2.warpAffine(mask,  M, (w, h), flags=cv2.INTER_NEAREST)

    # Brightness / contrast
    if random.random() > 0.3:
        alpha = 0.7 + random.random() * 0.6   # wider range: 0.7–1.3
        beta  = random.randint(-20, 20)         # stronger shift
        image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    # Gaussian noise
    if random.random() > 0.3:
        noise = np.random.normal(0, random.uniform(2, 25), image.shape).astype(np.float32)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # Gaussian blur
    if random.random() > 0.3:
        ksize = random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ksize, ksize), 0)

    # Elastic deformation — critical for small medical datasets
    if random.random() > 0.5:
        h, w   = image.shape[:2]
        sigma  = random.uniform(6, 10)
        alpha  = random.uniform(30, 60)
        dx = cv2.GaussianBlur(
            np.random.randn(h, w).astype(np.float32), (0, 0), sigma
        ) * alpha
        dy = cv2.GaussianBlur(
            np.random.randn(h, w).astype(np.float32), (0, 0), sigma
        ) * alpha
        x, y   = np.meshgrid(np.arange(w), np.arange(h))
        map_x  = np.clip(x + dx, 0, w - 1).astype(np.float32)
        map_y  = np.clip(y + dy, 0, h - 1).astype(np.float32)
        image  = cv2.remap(image, map_x, map_y, cv2.INTER_LINEAR)
        mask   = cv2.remap(mask,  map_x, map_y, cv2.INTER_NEAREST)

    # Coarse dropout — randomly blank out patches (simulates ultrasound artefacts)
    if random.random() > 0.5:
        h, w     = image.shape[:2]
        n_holes  = random.randint(1, 4)
        hole_size = random.randint(16, 40)
        for _ in range(n_holes):
            x1 = random.randint(0, w - hole_size)
            y1 = random.randint(0, h - hole_size)
            image[y1:y1+hole_size, x1:x1+hole_size] = 0

    return image, mask
    

class BreastUltrasoundDataset(Dataset):

    def __init__(self, image_size=256, augment=True):

        self.root_dir = kagglehub.dataset_download(
            "aryashah2k/breast-ultrasound-images-dataset"
        )

        self.image_size = image_size
        self.samples = []
        self.do_augment = augment
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        
        classes = ["benign", "malignant"]

        for cls in classes:

            class_dir = os.path.join(
                self.root_dir,
                "Dataset_BUSI_with_GT",
                cls
            )

            files = os.listdir(class_dir)

            image_files = [
                f for f in files
                if "_mask" not in f and f.endswith(".png")
            ]

            for img_file in image_files:

                img_path = os.path.join(class_dir, img_file)

                base_name = img_file.replace(".png", "")

                mask_files = [
                    f for f in files
                    if f.startswith(base_name + "_mask")
                    and f.endswith(".png")
                ]

                mask_paths = [
                    os.path.join(class_dir, f)
                    for f in mask_files
                ]

                self.samples.append({
                    "image": img_path,
                    "masks": mask_paths,
                    "label": cls
                })

    def __len__(self):

        return len(self.samples)

    def pad_to_square(self, image, mask=None):

        h, w = image.shape[:2]

        size = max(h, w)

        pad_h = size - h
        pad_w = size - w

        top = pad_h // 2
        bottom = pad_h - top

        left = pad_w // 2
        right = pad_w - left

        image = cv2.copyMakeBorder(
            image,
            top,
            bottom,
            left,
            right,
            borderType=cv2.BORDER_CONSTANT,
            value=0
        )

        if mask is not None:

            mask = cv2.copyMakeBorder(
                mask,
                top,
                bottom,
                left,
                right,
                borderType=cv2.BORDER_CONSTANT,
                value=0
            )

            return image, mask

        return image

    def combine_masks(self, mask_paths, image_shape): 

        if len(mask_paths) == 0:

            return np.zeros(image_shape, dtype=np.uint8)

        combined_mask = np.zeros(image_shape, dtype=np.uint8)

        for path in mask_paths:

            mask = cv2.imread(
                path,
                cv2.IMREAD_GRAYSCALE
            )

            if mask is None:
                continue

            mask = (mask > 0).astype(np.uint8)

            combined_mask = np.maximum(
                combined_mask,
                mask
            )

        return combined_mask


    def __getitem__(self, idx):

        sample = self.samples[idx]

        image = cv2.imread(sample["image"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise ValueError(f"Failed to load image: {sample['image']}")

        mask = self.combine_masks(sample["masks"], image.shape)

        image, mask = self.pad_to_square(image, mask)

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        
        #CLAHE Preprocessing
        image = self.clahe.apply(image)
        
        if self.do_augment:
            image, mask = augment(image, mask)
            
        # normalize image
        image = image.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)


        mask = mask[None, :, :]   # (1, H, W)

        image = image[None, :, :] # (1, H, W)

        return torch.from_numpy(image).float(), torch.from_numpy(mask).float()

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):

        super().__init__()

        self.smooth = smooth

    def forward(self, preds, targets):

        preds   = torch.sigmoid(preds)
        preds   = preds.view(preds.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (preds * targets).sum(dim=1)

        dice = (
            2. * intersection + self.smooth
        ) / (
            preds.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )

        return 1 - dice.mean()


class FocalLoss(nn.Module):

    def __init__(self, alpha=0.5, gamma=2):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        bce = F.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none'
        )

        pt    = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce

        return focal.mean()

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        # alpha = FP weight, beta = FN weight
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        preds = preds.view(preds.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        TP = (preds * targets).sum(dim=1)
        FP = (preds * (1 - targets)).sum(dim=1)
        FN = ((1 - preds) * targets).sum(dim=1)
        tversky = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        return 1 - tversky.mean()


dice_loss  = DiceLoss()
focal_loss = FocalLoss()
tversky_loss = TverskyLoss() #Penalize FN

def criterion(preds, targets):

    dice  = dice_loss(preds, targets)
    focal = focal_loss(preds, targets)
    tversky = tversky_loss(preds, targets)
    return 0.7 * tversky +  0.3 * focal



In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [5]:
dataset = BreastUltrasoundDataset()
unet = UNet(1).to(device)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [6]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

train_dataset = BreastUltrasoundDataset(augment=True)
val_dataset   = BreastUltrasoundDataset(augment=False)
indices = list(range(len(train_dataset)))
labels = [s["label"] for s in train_dataset.samples]
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels
)

train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(Subset(val_dataset,   val_idx),   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

In [7]:
# =============================================================
# PARTICLE SWARM OPTIMISATION — Hyperparameter Search
# =============================================================
# Searches over 10 hyperparameters:
#   [0] encoder_lr      log-uniform  [1e-6, 1e-4]
#   [1] decoder_lr      log-uniform  [1e-5, 1e-3]
#   [2] encoder_wd      log-uniform  [1e-5, 1e-2]
#   [3] decoder_wd      log-uniform  [1e-5, 1e-2]
#   [4] dropout_p       uniform      [0.1,  0.6]
#   [5] tversky_weight  uniform      [0.4,  0.9]   (focal gets 1-w)
#   [6] focal_alpha     uniform      [0.25, 0.75]
#   [7] focal_gamma     uniform      [1.0,  4.0]
#   [8] tversky_alpha   uniform      [0.1,  0.5]   (FP penalty)
#   [9] tversky_beta    uniform      [0.5,  0.9]   (FN penalty)
#        Note: tversky_alpha + tversky_beta are kept <= 1 via clipping
#
# Each particle trains for PSO_EPOCHS epochs and is scored by
# validation Dice loss (lower = better).
# Best position stored in `pso_best_params`.
# =============================================================

import numpy as np

# ── PSO configuration ────────────────────────────────────────
N_PARTICLES   = 8       # swarm size (increase for more thorough search)
N_ITERATIONS  = 5       # PSO iterations
PSO_EPOCHS    = 5       # training epochs per particle evaluation
W_INERTIA     = 0.7     # inertia weight
C1_COGNITIVE  = 1.5     # personal-best attraction
C2_SOCIAL     = 1.5     # global-best attraction

# ── Search bounds ─────────────────────────────────────────────
# Dims 0-3: log10 of actual LR/WD values
# Dims 4-9: raw values
LOG_LOWER = np.array([-6.0, -5.0, -5.0, -5.0,  0.10, 0.40, 0.25, 1.0, 0.10, 0.50])
LOG_UPPER = np.array([-4.0, -3.0, -2.0, -2.0,  0.60, 0.90, 0.75, 4.0, 0.50, 0.90])
N_DIM = len(LOG_LOWER)

def decode(pos):
    """Convert raw PSO position → actual hyperparameter dict."""
    p = np.clip(pos, LOG_LOWER, LOG_UPPER)
    # Ensure tversky_alpha + tversky_beta <= 1 (rescale if needed)
    t_alpha = float(p[8])
    t_beta  = float(p[9])
    total   = t_alpha + t_beta
    if total > 1.0:
        t_alpha /= total
        t_beta  /= total
    return {
        "encoder_lr":     10 ** float(p[0]),
        "decoder_lr":     10 ** float(p[1]),
        "encoder_wd":     10 ** float(p[2]),
        "decoder_wd":     10 ** float(p[3]),
        "dropout_p":      float(p[4]),
        "tversky_weight": float(p[5]),
        "focal_alpha":    float(p[6]),
        "focal_gamma":    float(p[7]),
        "tversky_alpha":  t_alpha,
        "tversky_beta":   t_beta,
    }

def evaluate_particle(hp: dict, epochs: int = PSO_EPOCHS) -> float:
    """
    Build a fresh model + losses with the given hyperparameters,
    train for `epochs` epochs, return mean validation Dice loss.
    """
    model = UNet(n_class=1, dropout_p=hp["dropout_p"]).to(device)

    dec_params = [p for n, p in model.named_parameters() if "encoder" not in n]
    opt = torch.optim.AdamW([
        {"params": model.encoder.parameters(),
         "lr": hp["encoder_lr"], "weight_decay": hp["encoder_wd"]},
        {"params": dec_params,
         "lr": hp["decoder_lr"], "weight_decay": hp["decoder_wd"]},
    ])

    # Per-particle loss instances with tuned parameters
    _focal   = FocalLoss(alpha=hp["focal_alpha"],   gamma=hp["focal_gamma"])
    _tversky = TverskyLoss(alpha=hp["tversky_alpha"], beta=hp["tversky_beta"])
    _tw      = hp["tversky_weight"]

    def _criterion(preds, targets):
        return _tw * _tversky(preds, targets) + (1 - _tw) * _focal(preds, targets)

    _dice    = DiceLoss()
    _scaler  = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_val = float("inf")
    for _ in range(epochs):
        # — train —
        model.train()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            opt.zero_grad()
            if _scaler:
                with torch.amp.autocast("cuda"):
                    loss = _criterion(model(imgs), masks)
                _scaler.scale(loss).backward()
                _scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                _scaler.step(opt)
                _scaler.update()
            else:
                loss = _criterion(model(imgs), masks)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

        # — validate —
        model.eval()
        val_dice = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                if device.type == "cuda":
                    with torch.amp.autocast("cuda"):
                        out = model(imgs)
                else:
                    out = model(imgs)
                val_dice += _dice(out, masks).item()
        val_dice /= len(val_loader)
        if val_dice < best_val:
            best_val = val_dice

    del model, opt, _focal, _tversky
    torch.cuda.empty_cache()
    return best_val


# ── Initialise swarm ─────────────────────────────────────────
rng = np.random.default_rng(42)

positions  = rng.uniform(LOG_LOWER, LOG_UPPER, size=(N_PARTICLES, N_DIM))
velocities = rng.uniform(
    -(LOG_UPPER - LOG_LOWER) * 0.1,
     (LOG_UPPER - LOG_LOWER) * 0.1,
    size=(N_PARTICLES, N_DIM)
)

pbest_pos  = positions.copy()
pbest_val  = np.full(N_PARTICLES, float("inf"))
gbest_pos  = positions[0].copy()
gbest_val  = float("inf")

print("\n" + "="*65)
print("PSO — Hyperparameter Optimisation (10 dimensions)")
print(f"  Particles : {N_PARTICLES}  |  Iterations : {N_ITERATIONS}  |  Epochs/eval : {PSO_EPOCHS}")
print("="*65)

# ── PSO main loop ────────────────────────────────────────────
for iteration in range(N_ITERATIONS):
    print(f"\n── Iteration {iteration+1}/{N_ITERATIONS} ──")

    for i in range(N_PARTICLES):
        hp    = decode(positions[i])
        score = evaluate_particle(hp)

        print(
            f"  P{i:02d}  dice_loss={score:.4f}  "
            f"enc_lr={hp['encoder_lr']:.2e}  dec_lr={hp['decoder_lr']:.2e}  "
            f"drop={hp['dropout_p']:.2f}  tw={hp['tversky_weight']:.2f}  "
            f"f_α={hp['focal_alpha']:.2f}  f_γ={hp['focal_gamma']:.2f}  "
            f"tv_α={hp['tversky_alpha']:.2f}  tv_β={hp['tversky_beta']:.2f}"
        )

        if score < pbest_val[i]:
            pbest_val[i] = score
            pbest_pos[i] = positions[i].copy()

        if score < gbest_val:
            gbest_val = score
            gbest_pos = positions[i].copy()

    # ── Velocity & position update ───────────────────────────
    r1 = rng.uniform(0, 1, size=(N_PARTICLES, N_DIM))
    r2 = rng.uniform(0, 1, size=(N_PARTICLES, N_DIM))

    velocities = (
        W_INERTIA    * velocities
        + C1_COGNITIVE * r1 * (pbest_pos - positions)
        + C2_SOCIAL    * r2 * (gbest_pos - positions)
    )

    v_max      = 0.20 * (LOG_UPPER - LOG_LOWER)
    velocities = np.clip(velocities, -v_max, v_max)
    positions  = np.clip(positions + velocities, LOG_LOWER, LOG_UPPER)

    print(f"  → Global best so far: dice_loss={gbest_val:.4f}")


# ── Report best hyperparameters ──────────────────────────────
pso_best_params = decode(gbest_pos)

print("\n" + "="*65)
print("PSO COMPLETE — Best hyperparameters found:")
for k, v in pso_best_params.items():
    fmt = f"  {k:20s}: {v:.4e}" if v < 0.01 else f"  {k:20s}: {v:.4f}"
    print(fmt)
print(f"  {'best_dice_loss':20s}: {gbest_val:.4f}")
print(f"  {'best_dice_score':20s}: {1 - gbest_val:.4f}")
print("="*65)



PSO — Hyperparameter Optimisation (10 dimensions)
  Particles : 8  |  Iterations : 5  |  Epochs/eval : 5

── Iteration 1/5 ──


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  P00  dice_loss=0.7752  enc_lr=3.53e-05  dec_lr=7.55e-05  drop=0.15  tw=0.89  f_α=0.63  f_γ=3.36  tv_α=0.15  tv_β=0.68


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  P01  dice_loss=0.7175  enc_lr=5.52e-06  dec_lr=7.14e-04  drop=0.32  tw=0.51  f_α=0.53  f_γ=1.19  tv_α=0.36  tv_β=0.64
  P02  dice_loss=0.8092  enc_lr=3.28e-05  dec_lr=5.12e-05  drop=0.49  tw=0.50  f_α=0.48  f_γ=1.13  tv_α=0.16  tv_β=0.77


...

  P05  dice_loss=0.7527  enc_lr=6.95e-05  dec_lr=3.04e-04  drop=0.21  tw=0.42  f_α=0.29  f_γ=1.54  tv_α=0.26  tv_β=0.74


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  P06  dice_loss=0.4640  enc_lr=4.01e-05  dec_lr=1.00e-03  drop=0.39  tw=0.41  f_α=0.26  f_γ=2.34  tv_α=0.28  tv_β=0.72


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  P07  dice_loss=0.4760  enc_lr=8.11e-05  dec_lr=1.00e-03  drop=0.43  tw=0.47  f_α=0.31  f_γ=1.12  tv_α=0.28  tv_β=0.72
  → Global best so far: dice_loss=0.3981

PSO COMPLETE — Best hyperparameters found:
  encoder_lr          : 4.2576e-05
  decoder_lr          : 1.0000e-03
  encoder_wd          : 8.3114e-05
  decoder_wd          : 2.4489e-04
  dropout_p           : 0.3842
  tversky_weight      : 0.4875
  focal_alpha         : 0.2946
  focal_gamma         : 1.5136
  tversky_alpha       : 0.2550
  tversky_beta        : 0.7450
  best_dice_loss      : 0.3981
  best_dice_score     : 0.6019


In [9]:
# =============================================================
# Apply PSO-optimised hyperparameters to the final model
# =============================================================

unet = UNet(n_class=1, dropout_p=pso_best_params["dropout_p"]).to(device)

decoder_params = [p for n, p in unet.named_parameters() if "encoder" not in n]

optimizer = torch.optim.AdamW([
    {"params": unet.encoder.parameters(),
     "lr": pso_best_params["encoder_lr"],
     "weight_decay": pso_best_params["encoder_wd"]},
    {"params": decoder_params,
     "lr": pso_best_params["decoder_lr"],
     "weight_decay": pso_best_params["decoder_wd"]},
])

# Rebuild loss instances with PSO-optimised internal parameters
focal_loss   = FocalLoss(
    alpha=pso_best_params["focal_alpha"],
    gamma=pso_best_params["focal_gamma"]
)
tversky_loss = TverskyLoss(
    alpha=pso_best_params["tversky_alpha"],
    beta=pso_best_params["tversky_beta"]
)

_tw = pso_best_params["tversky_weight"]
def criterion(preds, targets):
    return _tw * tversky_loss(preds, targets) + (1 - _tw) * focal_loss(preds, targets)

# Rebuild scheduler for full 200-epoch run
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=10
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=190, eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[10]
)

scaler = torch.amp.GradScaler("cuda")

print("Model & losses rebuilt with PSO-optimised hyperparameters.")
print(f"  encoder_lr     : {pso_best_params['encoder_lr']:.2e}")
print(f"  decoder_lr     : {pso_best_params['decoder_lr']:.2e}")
print(f"  encoder_wd     : {pso_best_params['encoder_wd']:.2e}")
print(f"  decoder_wd     : {pso_best_params['decoder_wd']:.2e}")
print(f"  dropout_p      : {pso_best_params['dropout_p']:.4f}")
print(f"  tversky_weight : {pso_best_params['tversky_weight']:.4f}")
print(f"  focal_alpha    : {pso_best_params['focal_alpha']:.4f}")
print(f"  focal_gamma    : {pso_best_params['focal_gamma']:.4f}")
print(f"  tversky_alpha  : {pso_best_params['tversky_alpha']:.4f}  (FP penalty)")
print(f"  tversky_beta   : {pso_best_params['tversky_beta']:.4f}  (FN penalty)")


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Model & losses rebuilt with PSO-optimised hyperparameters.
  encoder_lr     : 4.26e-05
  decoder_lr     : 1.00e-03
  encoder_wd     : 8.31e-05
  decoder_wd     : 2.45e-04
  dropout_p      : 0.3842
  tversky_weight : 0.4875
  focal_alpha    : 0.2946
  focal_gamma    : 1.5136
  tversky_alpha  : 0.2550  (FP penalty)
  tversky_beta   : 0.7450  (FN penalty)


In [9]:
# checkpoint = torch.load("/kaggle/working/best_unet.pth", map_location=device)
# unet.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# scaler.load_state_dict(checkpoint["scaler_state_dict"])
# start_epoch = checkpoint["epoch"] + 1
# best_val    = checkpoint["best_val_dice_loss"]

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=200, eta_min=1e-6
# )
# scheduler.step(start_epoch) 

In [10]:
import torch.nn.functional as F

dice_criterion = DiceLoss()

num_epochs   = 200
patience     = 60
counter      = 0
train_losses = []
val_losses   = []
best_val     = 1

for epoch in range(num_epochs):

    # ===================== TRAIN =====================
    unet.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            outputs = unet(images)
            loss    = criterion(outputs, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ===================== VALIDATION =====================
    unet.eval()
    val_loss = 0
    dice     = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device)
            with torch.amp.autocast("cuda"):
                outputs = unet(images)
            val_loss += criterion(outputs, masks).item()
            dice     += dice_criterion(outputs, masks).item()

    avg_val_loss  = val_loss / len(val_loader)
    avg_dice_loss = dice     / len(val_loader)
    val_losses.append(avg_val_loss)

    if avg_dice_loss < best_val:
        best_val = avg_dice_loss
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     unet.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict":    scaler.state_dict(),
            "best_val_dice_loss":   best_val,
        }, "/kaggle/working/best_unet.pth")
        print(f"  Saved best model at epoch {epoch+1}")
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

    # ===================== LOG =====================
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train: {avg_train_loss:.4f} "
        f"Val: {avg_val_loss:.4f} "
        f"Dice Score: {1 - avg_dice_loss:.4f} "
        f"LR: {current_lr:.2e}"
    )

  Saved best model at epoch 1
Epoch [1/200] Train: 0.4337 Val: 0.4317 Dice Score: 0.1157 LR: 1.90e-04
  Saved best model at epoch 2
Epoch [2/200] Train: 0.4238 Val: 0.4064 Dice Score: 0.1575 LR: 2.80e-04
  Saved best model at epoch 3
Epoch [3/200] Train: 0.3687 Val: 0.3494 Dice Score: 0.2347 LR: 3.70e-04
  Saved best model at epoch 4
Epoch [4/200] Train: 0.3395 Val: 0.3379 Dice Score: 0.2585 LR: 4.60e-04
  Saved best model at epoch 5
Epoch [5/200] Train: 0.3156 Val: 0.3064 Dice Score: 0.3144 LR: 5.50e-04
  Saved best model at epoch 6
Epoch [6/200] Train: 0.2891 Val: 0.3079 Dice Score: 0.3413 LR: 6.40e-04
  Saved best model at epoch 7
Epoch [7/200] Train: 0.2638 Val: 0.2624 Dice Score: 0.4335 LR: 7.30e-04
  Saved best model at epoch 8
Epoch [8/200] Train: 0.2348 Val: 0.2316 Dice Score: 0.4941 LR: 8.20e-04
  Saved best model at epoch 9
Epoch [9/200] Train: 0.2181 Val: 0.1973 Dice Score: 0.5646 LR: 9.10e-04
  Saved best model at epoch 10
Epoch [10/200] Train: 0.2056 Val: 0.1966 Dice Score

In [ ]:
torch.save(unet.state_dict(), "model.bin")